In [8]:
from QualtricsAPI.Setup import Credentials
from QualtricsAPI.XM import XMDirectory
from QualtricsAPI.XM import MailingList
from QualtricsAPI.Survey import Responses
import shutil
import os
import requests
import zipfile
import json
import io
import pandas as pd
import re
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from reportlab.lib.pagesizes import letter, landscape, A4, A3
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, PageBreak, Paragraph, Spacer, Image, HRFlowable
from reportlab.lib import colors
from matplotlib.backends.backend_pdf import PdfPages
from reportlab.lib.enums import TA_CENTER, TA_RIGHT, TA_LEFT, TA_JUSTIFY
from reportlab.platypus import Paragraph, Spacer, KeepTogether, KeepInFrame 
from reportlab.graphics.shapes import Drawing, Rect, String
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from variableUtils import *
from Utils import *
pd.set_option('display.max_columns', None)

In [9]:
apiToken = 'HbeDqDB2kRbueZdGg9EhjMt2DDElVuvBxTR7HKhU'
dataCenter = 'syd1'
#Get Survey Responses (Updated)
kwargs = {"token": apiToken, "data_center": dataCenter}
Credentials().qualtrics_api_credentials(**kwargs)

surveyDict = {
              # 'Oral Med': 'SV_cON7eSWP0BaEfxs',
              # 'Paeds': 'SV_e9VZvANQN30lKzI',
              'Ortho': 'SV_5ySXHQWpXGSbucK',
              'Perio': 'SV_7VBSuXV72s6hHvg',
              'Endo': 'SV_9Tam9mLwlSIW4NU',
              'Pros': 'SV_43eYaPYStmWegvA',
              # "DCD Endo Seminar": "SV_6QouNmDXIbUFXOS",
              # "DCD Endo Lit": "SV_7W1LelzRLsmPG1E",
              # "DCD Endo Case": "SV_a3mtKAPnrX8DNvE"
}


dfDict = {}
for key, value in surveyDict.items():
    df = Responses().get_survey_responses(survey=value, verify=None, useLabels=True)
    # drop rows 0 and 1
    df.drop([0, 1], inplace=True)
    dfDict[key] = df
    # display(df.head(10))

In [10]:
# dfOmed = dfDict['Oral Med']
# dfPaeds = dfDict['Paeds']
dfOrtho = dfDict['Ortho']
dfPerio = dfDict['Perio']
dfEndo = dfDict['Endo']
dfPros = dfDict['Pros']
# print(len(dfOrtho))
# display(dfEndo.tail(5))
# dfEndo.to_excel('DCD/Endo.xlsx', index=False)
# display(dfPros.tail(5))


In [12]:
rubricColumns = ["Professionalism", "Dialogue", "Assessment", "Diff Diagnosis", "Treatment Plan", "Intervention", "Evaluation"]
readinessCol = 'Readiness to practic' # Level 1 etc.
colFinished = 'Finished' # True/False
siteCols = ["Site_4", "Site_5", "Site_10", "Site_11", "Site_12"]
sites = ["Royal Dental Hospital Melbourne RDHM", "Melbourne Dental Clinic", "RCH", "On call at RCH", "Shadowing at RCH"]

grades = ['Meets expectations at this point in education', "Does not meet expectations at this point in education", "Not applicable"]
strengths = 'Q11' #comments
weaknesses = 'Q12' #comments
# some rubrics have names instead of levels in readiness to practice, so we will need to map them to levels
rubricScoreDict = {
 'Critically behind expectations at this point in education': 'Level 1',
    'Slightly behind expectations at this point in education': 'Level 2',
    'Progressing appropriately': 'Level 3',
    'Surpassing expectations at this point in education': 'Level 4', 
}
# dfEndo[readinessCol] = dfEndo[readinessCol].replace(rubricScoreDict)

def getRubricCounts(studentDf, columnName):
    valueSeries = studentDf[columnName].fillna("").astype(str).str.strip()

    belowCount = (valueSeries == "Does not meet expectations at this point in education").sum()
    meetsCount = (valueSeries == "Meets expectations at this point in education").sum()
    naCount = (valueSeries == "Not applicable").sum()

    totalCount = belowCount + meetsCount + naCount

    return {
        "below": int(belowCount),
        "meets": int(meetsCount),
        "na": int(naCount),
        "total": int(totalCount)
    }

def createRubricBar(columnName, counts, barWidth=540, barHeight=24):
    drawingHeight = 58
    drawing = Drawing(barWidth, drawingHeight)
    belowColor = colors.HexColor("#094183")
    meetsColor = colors.HexColor("#4597ad")
    naColor = colors.HexColor("#6f6f6f")
    borderColor = colors.HexColor("#0f0f0f")
    emptyColor = colors.white
    maxCells = counts["total"] if counts["total"] > 0 else 1

    labelFontSize = 9
    titleFontSize = 11

    if maxCells is None:
        maxCells = counts["total"] if counts["total"] > 0 else 1

    drawingHeight = 60
    drawing = Drawing(barWidth, drawingHeight)

    drawing.add(String(0, 46, columnName, fontName="Helvetica-Bold", fontSize=titleFontSize))
    # drawing.add(String(0, 28, "Student self-evaluation", fontName="Helvetica", fontSize=10))
    drawing.add(String(barWidth/6, 8 + barHeight + 10, "Below Expectation", fontName="Helvetica-Bold", fontSize=labelFontSize))
    # add a small rectangle as a legend for below expectation
    drawing.add(Rect(barWidth/6 - 12, 8 + barHeight + 6, 10, 10, fillColor=belowColor, strokeColor=borderColor, strokeWidth=0.8))
    drawing.add(String(barWidth/2, 8 + barHeight + 10, "Meets Expectation", fontName="Helvetica-Bold", fontSize=labelFontSize))
    # add a small rectangle as a legend for meets expectation
    drawing.add(Rect(barWidth/2 - 12, 8 + barHeight + 6, 10, 10, fillColor=meetsColor, strokeColor=borderColor, strokeWidth=0.8))
    drawing.add(String(5*barWidth/6, 8 + barHeight + 10, "Not Applicable", fontName="Helvetica-Bold", fontSize=labelFontSize))
    # add a small rectangle as a legend for not applicable
    drawing.add(Rect(5*barWidth/6 - 12, 8 + barHeight + 6, 10, 10, fillColor=naColor, strokeColor=borderColor, strokeWidth=0.8))

    barY = 8
    cellWidth = barWidth / maxCells

    cellColors = ([belowColor] * counts["below"] + [meetsColor] * counts["meets"] + [naColor] * counts["na"])

    if len(cellColors) < maxCells:
        cellColors += [emptyColor] * (maxCells - len(cellColors))

    for index in range(maxCells):
        xPos = index * cellWidth
        drawing.add(Rect(xPos, barY, cellWidth, barHeight, fillColor=cellColors[index], strokeColor=borderColor, strokeWidth=0.8))

    drawing.add(Rect(0, barY, barWidth, barHeight, fillColor=None, strokeColor=borderColor, strokeWidth=1))

    return drawing

def createReadinessLegend():
    drawing = Drawing(520, 20)

    items = [
        ("Level 1", colors.HexColor("#c0392b")), ("Level 2", colors.HexColor("#e67e22")), 
        ("Level 3", colors.HexColor("#56a3b9")), ("Level 4", colors.HexColor("#4c8f5a")),
    ]

    x = 0
    for label, color in items:
        drawing.add(Rect(x, 4, 12, 12, fillColor=color, strokeColor=colors.black))
        drawing.add(String(x + 18, 6, label, fontName="Helvetica", fontSize=9))
        x += 120

    return drawing

def createReadinessBar(studentDf, readinessCol, barWidth=520, barHeight=22):

    levelSeries = (
        studentDf[readinessCol]
        .astype(str)
        .str.extract(r"Level\s*(\d+)", expand=False)
        .dropna()
        .astype(int)
    )

    counts = levelSeries.value_counts().to_dict()

    maxLevel = 4
    maxCells = max(len(levelSeries), 1)

    levelColors = {
        1: colors.HexColor("#c0392b"),
        2: colors.HexColor("#e67e22"),
        3: colors.HexColor("#56a3b9"),
        4: colors.HexColor("#4c8f5a")
    }

    borderColor = colors.HexColor("#666666")

    drawing = Drawing(barWidth, 60)

    drawing.add(String(0, 42, "Readiness to Practice", fontName="Helvetica-Bold", fontSize=14))

    cellWidth = barWidth / maxCells
    barY = 8

    levelCells = []

    for level in range(1, maxLevel + 1):
        levelCells += [level] * counts.get(level, 0)

    if len(levelCells) < maxCells:
        levelCells += [0] * (maxCells - len(levelCells))

    for i, level in enumerate(levelCells):

        x = i * cellWidth

        fill = levelColors.get(level, colors.white)

        drawing.add(Rect(x, barY, cellWidth, barHeight, fillColor=fill, strokeColor=borderColor, strokeWidth=0.8))

    drawing.add(Rect(0, barY, barWidth, barHeight, fillColor=None, strokeColor=borderColor, strokeWidth=1))

    return drawing

def createStudentReport(df, studentname, folder, cohortname):
    subheadingStyleL.fontSize = 18
    doc = SimpleDocTemplate(f"{folder}/{studentname}.pdf", pagesize=pageSize, rightMargin=rightMargin, leftMargin=leftMargin, topMargin=topMargin, bottomMargin=bottomMargin)
    elements = []
    studentDf = df[df['Student Name'] == studentname]
    # display(studentDf)
    if studentDf.empty:
        print(f"No data found for student: {studentname}")
        return None
    elements.append(Spacer(1, 72))

    # add number of finshed forms
    elements.append(Paragraph(f"Number of feedback forms: {len(studentDf)}", subheadingStyleL))

    # add clinical placements
    elements.append(Paragraph("Clinical placements", subheadingStyleL))
    allSites = studentDf['Sites Attended'].str.split(', ').explode().unique()
    allSites = [site for site in allSites if pd.notna(site)]
    allSites = list(set(allSites))
    siteRows = []
    for site in sites:
        marker = "[X]" if site in allSites else "[ ]"
        siteRows.append([Paragraph(marker, subsubheadingStyleL), Paragraph(site, subsubheadingStyleL)])
    siteTable = Table(siteRows, colWidths=[0.3*inch, 5*inch], hAlign='LEFT')
    siteTable.setStyle(TableStyle([
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('LEFTPADDING', (0, 0), (-1, -1), 0),
        ('RIGHTPADDING', (0, 0), (-1, -1), 0),
        ('TOPPADDING', (0, 0), (-1, -1), 2),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 2),
    ]))
    elements.append(siteTable)
    elements.append(Spacer(1, 36))

    # add rubric bars
    elements.append(Paragraph("Feedback Domains", subheadingStyleL))
    elements.append(Spacer(1, 12))
    for rubricCol in rubricColumns:
        if rubricCol in studentDf.columns:
            counts = getRubricCounts(studentDf, rubricCol)
            rubricBar = createRubricBar(rubricCol, counts, barWidth = pageSize[0] - leftMargin - rightMargin - 2*inch, barHeight=18)
            elements.append(rubricBar)
            elements.append(Spacer(1, 12))

    # add readiness bar
    elements.append(HRFlowable(width="100%", thickness=1, color=uniColor))
    if readinessCol in studentDf.columns:
        readinessBar = createReadinessBar(studentDf, readinessCol, barWidth = pageSize[0] - leftMargin - rightMargin - 2*inch, barHeight=18)
        elements.append(Spacer(1, 12))
        elements.append(readinessBar)
        readinesslegend = createReadinessLegend()
        elements.append(Spacer(1, 6))
        elements.append(readinesslegend)

    # add comments   
    elements.append(PageBreak())
    elements.append(Paragraph("Strengths", subheadingStyleL))
    strengthDf = studentDf[['Date', strengths]]
    strengthDf = strengthDf.dropna(subset=[strengths])
    strengthDf.columns = ['Date', 'Strongest aspects of performance']
    strengthTable = createTable(strengthDf, title ="", colRatio = [1, 5], customTextCols=[0,1], tableTextStyle=subsubheadingStyleL, tableWidth = 0.8, headerColor = uniColor)
    elements.append(strengthTable)

    # for weaknesses now
    elements.append(Spacer(1, 12))
    weaknessDf = studentDf[['Date', weaknesses]]
    weaknessDf = weaknessDf.dropna(subset=[weaknesses])
    weaknessDf.columns = ['Date', 'Areas for improvement']
    weaknessTable = createTable(weaknessDf, title ="", colRatio = [1, 5], customTextCols=[0,1], tableTextStyle=subsubheadingStyleL, tableWidth = 0.8, headerColor = uniColor)
    elements.append(KeepTogether([Paragraph("Areas for Improvement", subheadingStyleL), weaknessTable]))

    doc.build(elements, onFirstPage=getBannerDrawer(f'DCD {cohortname} feedback', studentname))
    
def createCohortReport(df, cohortname):
    folder = f'DCD/{cohortname}'
    thisSiteCols = [col for col in siteCols if col in df.columns]
    print(f"Processing cohort: {cohortname}, Sites columns found: {thisSiteCols}")
    # merge the sites columns into one column called 'Sites Attended'
    display(df[thisSiteCols].head(5))
    df['Sites Attended'] = df[thisSiteCols].apply(lambda x: ', '.join(x.dropna()), axis=1)
    # strip 
    display(df['Sites Attended'])
    df['Sites Attended'] = df['Sites Attended'].str.strip()
    df[colFinished] = df[colFinished].astype(str)
    df = df[df[colFinished] == 'True']
    df["ReadinessLevel"] = (
    df["Readiness to practic"]
    .astype(str)
    .str.extract(r"Level\s*(\d+)", expand=False)
    .astype("Int64")
    )
    for name in df['Student Name'].unique():
        createStudentReport(df, name, folder, cohortname)
        # break # Remove this break to generate reports for all students


# createCohortReport(dfPaeds, 'Paeds')
# createCohortReport(dfOmed, 'Oral Med')
createCohortReport(dfOrtho, 'Ortho')
createCohortReport(dfPros, 'Pros')
createCohortReport(dfEndo, 'Endo')
# createCohortReport(dfPerio, 'Perio')

Processing cohort: Ortho, Sites columns found: ['Site_4', 'Site_5']


,Site_4,Site_5
2,NaN,Melbourne Dental Clinic
3,NaN,Melbourne Dental Clinic
4,NaN,Melbourne Dental Clinic
5,NaN,Melbourne Dental Clinic
6,NaN,Melbourne Dental Clinic


2      Melbourne Dental Clinic
3      Melbourne Dental Clinic
4      Melbourne Dental Clinic
5      Melbourne Dental Clinic
6      Melbourne Dental Clinic
                ...           
239    Melbourne Dental Clinic
240    Melbourne Dental Clinic
241    Melbourne Dental Clinic
242    Melbourne Dental Clinic
243    Melbourne Dental Clinic
Name: Sites Attended, Length: 242, dtype: object

Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 


C:\Users\Kunal Patel\AppData\Local\Temp\ipykernel_28084\587304607.py:234: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["ReadinessLevel"] = (


Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Processing cohort: Pros, Sites columns found: ['Site_4', 'Site_5']


... (6 outputs left out)

2     Royal Dental Hospital Melbourne RDHM
3     Royal Dental Hospital Melbourne RDHM
4     Royal Dental Hospital Melbourne RDHM
5     Royal Dental Hospital Melbourne RDHM
6     Royal Dental Hospital Melbourne RDHM
7     Royal Dental Hospital Melbourne RDHM
8     Royal Dental Hospital Melbourne RDHM
9     Royal Dental Hospital Melbourne RDHM
10    Royal Dental Hospital Melbourne RDHM
11    Royal Dental Hospital Melbourne RDHM
12    Royal Dental Hospital Melbourne RDHM
13    Royal Dental Hospital Melbourne RDHM
14    Royal Dental Hospital Melbourne RDHM
15    Royal Dental Hospital Melbourne RDHM
16    Royal Dental Hospital Melbourne RDHM
17    Royal Dental Hospital Melbourne RDHM
18    Royal Dental Hospital Melbourne RDHM
19    Royal Dental Hospital Melbourne RDHM
20    Royal Dental Hospital Melbourne RDHM
21    Royal Dental Hospital Melbourne RDHM
22    Royal Dental Hospital Melbourne RDHM
23    Royal Dental Hospital Melbourne RDHM
24    Royal Dental Hospital Melbourne RDHM
25    Royal

Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 


C:\Users\Kunal Patel\AppData\Local\Temp\ipykernel_28084\587304607.py:234: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["ReadinessLevel"] = (


Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 
Creating table for 


## Other reports

In [8]:
surveyDict['Case Report'] = 'SV_a3mtKAPnrX8DNvE'
surveyDict['Lit Review'] = 'SV_7W1LelzRLsmPG1E'
surveyDict['Seminar Skills'] = 'SV_6QouNmDXIbUFXOS'
dfCaseReport = Responses().get_survey_responses(survey=surveyDict['Case Report'], verify=None, useLabels=True)
dfLitReview = Responses().get_survey_responses(survey=surveyDict['Lit Review'], verify=None, useLabels=True)
dfSeminarSkills = Responses().get_survey_responses(survey=surveyDict['Seminar Skills'], verify=None, useLabels=True)

In [9]:
# save them to excel
dfCaseReport.to_excel('DCD/CaseReport.xlsx', index=False)
dfLitReview.to_excel('DCD/LitReview.xlsx', index=False)
dfSeminarSkills.to_excel('DCD/SeminarSkills.xlsx', index=False)

### Case Report / Lit Review / Seminar Skills — cleaned workbooks + per-student PDFs
Code lives in `dcd_skills_utils.py` (project root). Pipeline:
1. `loadAllDcdSkillsForms()` — reads the three Qualtrics exports (or the live `dfCaseReport` / `dfLitReview` / `dfSeminarSkills` from the cells above), parses domain ratings (`4-Very good` → 4), overall %, bands, normalises assessor names, harmonises DCD Year (modal per student) and tags excluded rows (survey previews, `Finished = False`, pre-25 Feb 2026 form-build tests). Nothing is deleted — excluded rows sit in the `Excluded` sheet.
2. `buildDcdSkillsWorkbooks()` — one styled workbook per form in `DCD/Skills/` (Summary / Forms / Comments / Excluded / Colour Key).
3. `buildDcdSkillsReports()` — per-student PDFs: `combined=True` → `DCD/Skills/Combined/<name>.pdf` (overview page + one section per form, page-break separated); `separate=True` → `DCD/Skills/<Form>/<name>.pdf`.

Tweak cleaning with `cleanConfig=dict(minDate='2026-02-25', testAssessors=(...), placeholderAssessors=('ABC',))`.


In [ ]:
from dcd_skills_utils import *

# Source: saved exports (default) — or pass the live frames pulled above:
# formDfs = loadAllDcdSkillsForms({'Case Report': dfCaseReport, 'Lit Review': dfLitReview, 'Seminar Skills': dfSeminarSkills})
formDfs = loadAllDcdSkillsForms()

# Quick look at the cleaned Case Report rows
display(keptRows(formDfs['Case Report']).head())


In [ ]:
# 1) Cleaned + colour-coded workbooks, one per form -> DCD/Skills/DCD_<Form>_cleaned.xlsx
workbookPaths = buildDcdSkillsWorkbooks(formDfs, outFolder='DCD/Skills')


In [ ]:
# 2) Per-student PDFs
#    combined=True  -> DCD/Skills/Combined/<student>.pdf  (overview + Case Report / Lit Review / Seminar Skills sections)
#    separate=True  -> DCD/Skills/<Form>/<student>.pdf    (one PDF per form)
#    students=['Adam Saker'] to limit; None = everyone with at least one completed form
pdfPaths = buildDcdSkillsReports(formDfs, outFolder='DCD/Skills', combined=True, separate=False, students=None)


In [ ]:
# Cross-form summary table (handy for Clare / coordinators)
summaryAll = pd.concat([studentSummary(df).assign(Form=k) for k, df in formDfs.items()], ignore_index=True)
display(summaryAll.pivot_table(index=['Student Name', 'DCD Year'], columns='Form',
                               values=['Forms', 'Overall Mean', 'Latest Overall'], aggfunc='first'))
